In [1]:
import pandas as pd
import numpy as np

pd.set_option("future.no_silent_downcasting", True)

### IMPORTS

In [2]:
# The Pudding's script gender analysis dataset:
gender_data = pd.read_csv("pudding/character_list5.csv", header=0, encoding="latin-1")
script_metadata = pd.read_csv("pudding/meta_data7.csv", header=0, encoding="latin-1")

# Merge for movie data
gender_raw = pd.merge(gender_data, script_metadata, how="left", on="script_id")
gender = gender_raw[["imdb_character_name", "words", "gender", "age", "title", "year"]].copy()

# Clean
gender.loc[:, "title"] = gender["title"].str.replace(r'[^A-Za-z0-9 ]', '', regex=True).str.lower().str.strip()

# Handle missing data
gender["gender"] = gender["gender"].replace("?", np.nan)
gender["gender"] = gender["gender"].astype("category")

# print(gender["gender"].isnull().sum())
gender = gender.dropna(subset=["gender"])

gender = gender.rename(columns={"imdb_character_name": "character"})

print(gender.info())

<class 'pandas.core.frame.DataFrame'>
Index: 23043 entries, 0 to 23047
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   character  23041 non-null  object  
 1   words      23043 non-null  int64   
 2   gender     23043 non-null  category
 3   age        18261 non-null  float64 
 4   title      23043 non-null  object  
 5   year       23043 non-null  int64   
dtypes: category(1), float64(1), int64(2), object(2)
memory usage: 1.1+ MB
None


In [3]:
# Oscars dataset:
oscars_raw = pd.read_csv("oscars/the_oscar_award.csv", header=0)
oscars = oscars_raw[["winner"]].copy()

# Clean
oscars["title"] = oscars_raw["film"].str.replace(r'[^A-Za-z0-9 ]', '', regex=True).str.lower().str.strip()
oscars["year"] = (oscars_raw["year_film"])
oscars["award"] = oscars_raw["canon_category"].astype("category")

# Handle missing data
oscars = oscars.dropna(subset=["title"])

print(oscars.info())

<class 'pandas.core.frame.DataFrame'>
Index: 10751 entries, 0 to 11105
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   winner  10751 non-null  bool    
 1   title   10751 non-null  object  
 2   year    10751 non-null  int64   
 3   award   10751 non-null  category
dtypes: bool(1), category(1), int64(1), object(1)
memory usage: 275.5+ KB
None


### DATA WRANGLING & TIDYING

In [4]:
# Aggregate & pivot
film_gender = (
    gender.groupby(["title", "year", "gender"], observed=True)
      .agg(total_words=("words", "sum"),
           num_characters=("character", "count"))
      .reset_index()
)

# Pivot for m/f columns
film_gender = film_gender.pivot(
    index=["title", "year"],
    columns="gender",
    values=["total_words", "num_characters"]
).fillna(0)

film_gender.columns = ['_'.join(col).strip().lower() for col in film_gender.columns.values]
film_gender = film_gender.reset_index()

# Calculate proportion of words by gender
film_gender["pct_words_f"] = np.where(
    (film_gender["total_words_m"] + film_gender["total_words_f"]) > 0,
    film_gender["total_words_f"] / (film_gender["total_words_m"] + film_gender["total_words_f"]),
    0
)

film_gender["pct_words_m"] = np.where(
    (film_gender["total_words_m"] + film_gender["total_words_f"]) > 0,
    film_gender["total_words_m"] / (film_gender["total_words_m"] + film_gender["total_words_f"]),
    0
)

# Calculate proportion of genders by character count
film_gender["pct_chars_f"] = np.where(
    (film_gender["num_characters_m"] + film_gender["num_characters_f"]) > 0,
    film_gender["num_characters_f"] / (film_gender["num_characters_m"] + film_gender["num_characters_f"]),
    0
)

film_gender["pct_chars_m"] = np.where(
    (film_gender["num_characters_m"] + film_gender["num_characters_f"]) > 0,
    film_gender["num_characters_m"] / (film_gender["num_characters_m"] + film_gender["num_characters_f"]),
    0
)

# Calculate average word count by gender
film_gender["avg_male_words"] = np.where(
    film_gender["num_characters_m"] > 0,
    film_gender["total_words_m"] / film_gender["num_characters_m"],
    0
)

film_gender["avg_female_words"] = np.where(
    film_gender["num_characters_f"] > 0,
    film_gender["total_words_f"] / film_gender["num_characters_f"],
    0
)

# Determine basic gender bias by average word count per gender
film_gender["dominant_gender"] = np.where(
    film_gender["avg_male_words"] >= film_gender["avg_female_words"],
    "m",
    "f"
)

print(film_gender.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             2000 non-null   object 
 1   year              2000 non-null   int64  
 2   total_words_f     2000 non-null   float64
 3   total_words_m     2000 non-null   float64
 4   num_characters_f  2000 non-null   float64
 5   num_characters_m  2000 non-null   float64
 6   pct_words_f       2000 non-null   float64
 7   pct_words_m       2000 non-null   float64
 8   pct_chars_f       2000 non-null   float64
 9   pct_chars_m       2000 non-null   float64
 10  avg_male_words    2000 non-null   float64
 11  avg_female_words  2000 non-null   float64
 12  dominant_gender   2000 non-null   object 
dtypes: float64(10), int64(1), object(2)
memory usage: 203.3+ KB
None


In [5]:
# Left join on title and year for Academy Award data to keep 0 nominations
oscars_gender = pd.merge(film_gender, oscars, on=["title", "year"], how="left")

# NAs in winner and award columns indicate no nomination
# print(oscars_gender.isna().sum())

# Split acting vs non-acting awards due to inherent gendering
if_act = oscars_gender["award"].str.contains("ACTOR|ACTRESS", case=False, na=False)
oscars_acting = oscars_gender[if_act]
oscars_other = oscars_gender[~if_act]

print(len(oscars_acting))
print(len(oscars_other))

517
3144


In [6]:
# Aggregate non-acting
oscars_summary = oscars_other.groupby(["title", "year"], observed=True).agg(
    total_nominations=("award", lambda x: x.notna().sum()),
    total_wins=("winner", lambda x: x.fillna(False).astype(bool).sum())
).reset_index()

# Merge back other columns, drop categories
oscars_summary = pd.merge(oscars_other, oscars_summary, on=["title", "year"], how="left").drop_duplicates(subset=["title", "year"])
oscars_summary = oscars_summary.drop(["winner", "award"], axis=1)

# Handle missing data
# print(oscars_summary.isnull().sum())
oscars_summary = oscars_summary.fillna({"total_nominations": 0, "total_wins": 0})

# Add nomination flag
oscars_summary["nominated"] = np.where(oscars_summary["total_nominations"] > 0, True, False)

# Add non-acting flag
oscars_summary = oscars_summary.assign(df_type = "Other")

print(oscars_summary.head())

                         title  year  total_words_f  total_words_m  \
0   10 things i hate about you  1999         8992.0        10688.0   
1               12 and holding  2005         5324.0        10644.0   
2             12 years a slave  2013         3452.0        16176.0   
8                    127 hours  2010          809.0         4336.0   
13                        1408  2007          284.0         3039.0   

    num_characters_f  num_characters_m  pct_words_f  pct_words_m  pct_chars_f  \
0                4.0               8.0     0.456911     0.543089     0.333333   
1                5.0              10.0     0.333417     0.666583     0.333333   
2                6.0              22.0     0.175871     0.824129     0.214286   
8                3.0               1.0     0.157240     0.842760     0.750000   
13               1.0               3.0     0.085465     0.914535     0.250000   

    pct_chars_m  avg_male_words  avg_female_words dominant_gender  \
0      0.666667     133

In [7]:
# Aggregate acting awards
acting_summary = oscars_acting.groupby(["title", "year", "dominant_gender"], observed=True).agg(
    # boolean acting
    actor_nom=("award", lambda x: (x.str.contains("ACTOR", case=False)).sum() > 0),
    actress_nom=("award", lambda x: (x.str.contains("ACTRESS", case=False)).sum() > 0),
    actor_win=("winner", lambda x: (
        x & oscars_acting.loc[x.index, "award"].str.contains("ACTOR", case=False, na=False)
    ).sum() > 0),
    actress_win=("winner", lambda x: (
        x & oscars_acting.loc[x.index, "award"].str.contains("ACTRESS", case=False, na=False)
    ).sum() > 0),
    # count acting
    actor_noms=("award", lambda x: (x.str.contains("ACTOR", case=False)).sum()),
    actress_noms=("award", lambda x: (x.str.contains("ACTRESS", case=False)).sum()),
    actor_wins=("winner", lambda x: (
        x & oscars_acting.loc[x.index, "award"].str.contains("ACTOR", case=False, na=False)
    ).sum()),
    actress_wins=("winner", lambda x: (
        x & oscars_acting.loc[x.index, "award"].str.contains("ACTRESS", case=False, na=False)
    ).sum())
).reset_index()

# Merge back other columns, drop categories
acting_summary = pd.merge(oscars_acting, acting_summary, on=["title", "year", "dominant_gender"], how="left").drop_duplicates(subset=["title", "year"])
acting_summary = acting_summary.drop(["winner", "award"], axis=1)

# Mark as acting
acting_summary = acting_summary.assign(df_type = "Acting")

print(acting_summary.info())

<class 'pandas.core.frame.DataFrame'>
Index: 315 entries, 0 to 516
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             315 non-null    object 
 1   year              315 non-null    int64  
 2   total_words_f     315 non-null    float64
 3   total_words_m     315 non-null    float64
 4   num_characters_f  315 non-null    float64
 5   num_characters_m  315 non-null    float64
 6   pct_words_f       315 non-null    float64
 7   pct_words_m       315 non-null    float64
 8   pct_chars_f       315 non-null    float64
 9   pct_chars_m       315 non-null    float64
 10  avg_male_words    315 non-null    float64
 11  avg_female_words  315 non-null    float64
 12  dominant_gender   315 non-null    object 
 13  actor_nom         315 non-null    bool   
 14  actress_nom       315 non-null    bool   
 15  actor_win         315 non-null    bool   
 16  actress_win       315 non-null    bool   
 17  ac

In [8]:
# Get character age data
gender_age = gender.dropna(subset=["age"])

# Remove unrealistic ages
gender_age = gender_age[gender_age["age"] <= 100]

# Merge with acting data
gender_age = pd.merge(gender_age, acting_summary, on=["title", "year"], how="inner")

# Drop extra columns
gender_age.drop(
    columns=['total_words_f', 'total_words_m', 'num_characters_f', 'num_characters_m', 'pct_words_f',
        'pct_words_m', 'pct_chars_f', 'pct_chars_m', 'avg_male_words', 'avg_female_words'],
        inplace=True)

print(gender_age.columns)

Index(['character', 'words', 'gender', 'age', 'title', 'year',
       'dominant_gender', 'actor_nom', 'actress_nom', 'actor_win',
       'actress_win', 'actor_noms', 'actress_noms', 'actor_wins',
       'actress_wins', 'df_type'],
      dtype='object')


In [9]:
# Save to CSV
oscars_summary.to_csv("oscars_summary.csv", index=False)
acting_summary.to_csv("acting_summary.csv", index=False)
gender_age.to_csv("gender_age.csv", index=False)